In [1]:
from IPython.display import HTML, display

html_content = """
<div style="font-family:Arial, sans-serif; padding:20px; border:2px solid #0078D4; border-radius:10px; background:#f8f9fa; width:85%;">

    <h2 style="color:#0078D4;">Two-Agent OpenAI System</h2>

    <p>
        A <b>Two-Agent System</b> consists of two AI agents working together.
        Instead of one AI doing everything, each agent has a specific responsibility.
    </p>

    <h3>Simple Analogy</h3>

    <p>
        Imagine visiting an IT Service Desk:
    </p>

    <ul>
        <li><b>Agent 1:</b> Front Desk Executive</li>
        <li><b>Agent 2:</b> Technical Expert</li>
    </ul>

    <p>
        The Front Desk Executive understands the request and forwards it to the Technical Expert.
        The Technical Expert provides the solution and sends it back to the user.
    </p>

    <h3>Example</h3>

    <div style="background:white; padding:10px; border-left:5px solid #0078D4;">
        <b>User:</b> I need GIS access.
    </div>

    <br>

    <div style="background:white; padding:10px; border-left:5px solid #28a745;">
        <b>Agent 1 - Request Analyzer</b><br>
        Identifies that this is an Access Request.
    </div>

    <br>

    <div style="background:white; padding:10px; border-left:5px solid #ff9800;">
        <b>Agent 2 - Knowledge Expert</b><br>
        Determines GIS requires Manager Approval and must be assigned to the GIS Support Team.
    </div>

    <br>

    <div style="background:white; padding:10px; border-left:5px solid #9c27b0;">
        <b>Final Response</b><br>
        Category: Access Request<br>
        Assignment Group: GIS Support Team<br>
        Approval Required: Manager Approval
    </div>

    <h3>Workflow</h3>

    <pre style="background:#eeeeee;padding:10px;">
User
 ↓
Agent 1
(Understand Request)
 ↓
Agent 2
(Solve / Research)
 ↓
Final Response
    </pre>

    <h3>Benefits</h3>

    <ul>
        <li>Work is divided between agents.</li>
        <li>Each agent specializes in a specific task.</li>
        <li>Produces more accurate answers.</li>
        <li>Handles complex tasks efficiently.</li>
    </ul>

    <div style="margin-top:15px;
                background:#d4edda;
                padding:15px;
                border-radius:5px;
                font-weight:bold;
                font-size:16px;">
        Two-Agent System = One AI understands the problem, another AI helps solve it.
    </div>

</div>
"""

display(HTML(html_content))

In [2]:
import autogen
autogen.__version__

'0.14.1'

In [3]:
keys = {}

with open("secrets.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        keys[key] = value

agentops_api_key = keys["AGENTOPS_API_KEY"]
openai_api_key = keys["OPENAI_API_KEY"]

In [4]:
#import agentops
#agentops.init(agentops_api_key)


In [5]:
from pathlib import Path

from autogen import ConversableAgent
from autogen.coding import  CodeBlock, LocalCommandLineCodeExecutor
work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

#Agent 1
code_executor_agent = ConversableAgent(
    name="code_executor_agent",
    llm_config=False,
    code_execution_config={
        "executor": executor,
    },
    human_input_mode="NEVER",
)

In [6]:
# The code writer agent's system message is to instruct the LLM on how to
# use the Jupyter code executor with IPython kernel.
code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
In the following cases, suggest python code (in a python coding block) or shell script (in a sh coding block) for the user 
to execute.
    1. When you need to collect info, use the code to output the info you need, for example, browse or search the web, 
    download/read a file, 
    print the content of a webpage or a file, get the current date/time, check the operating system. After sufficient 
    info is printed and the 
    task is ready to be solved based on your language skill, you can solve the task by yourself.
    2. When you need to perform some task with code, use the code to perform the task and output the result. Finish the
    task smartly.
Solve the task step by step if you need to. If a plan is not provided, explain your plan first. Be clear which step uses 
code, and which step uses your 
language skill.When using code, you must indicate the script type in the code block. The user cannot provide any other
feedback or perform any other 
action beyond executing the code you suggest. The user can't modify your code. So do not suggest incomplete code which
requires users to modify. 
Don't use a code block if it's not intended to be executed by the user.
If you want the user to save the code in a file before executing it, put # filename: <filename> inside the code block 
as the first line. Don't include multiple code blocks in one response. Do not ask users to copy and paste the result. 
Instead, use 'print' function for the output when relevant. Check the execution result returned by the user.
"""

import os

#Agent 2

code_writer_agent = ConversableAgent(
    "code_writer",
    system_message=code_writer_system_message,
    llm_config={
        "config_list": [
            {
                "model": "gpt-4",
                "api_key": openai_api_key
            }
        ]
    },
    code_execution_config=False,
    max_consecutive_auto_reply=2,
    human_input_mode="NEVER",
)

In [8]:
import pprint

chat_result = code_executor_agent.initiate_chat(
    code_writer_agent, message=" write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count."
)

pprint.pprint(chat_result)
#agentops.end_session("Success")

code_executor_agent (to code_writer):

 write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count.

--------------------------------------------------------------------------------
code_writer (to code_executor_agent):

Sure, we can use the permutations function from the itertools module of python to find the permutations of the word 'ALGEBRA'. 

Here is your python code:

```python
import itertools

def count_permutations(word):
    # Creating a set to store unique permutations
    unique_permutations = set(itertools.permutations(word))
    return len(unique_permutations)

word = 'ALGEBRA'
print(count_permutations(word))
```
This code will count the number of unique permutations of the word 'ALGEBRA'. It uses the itertools.permutations function to generate the permutations, and then a set is used to remove any duplicate permutations (since sets only contain unique elements). The len function is then used to count the number of

In [9]:
dir(chat_result)


['__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'chat_history',
 'chat_id',
 'cost',
 'human_input',
 'summary']

In [10]:
chat_result.chat_history

[{'content': ' write python code for permutation for the word ALGEbRA. use optimised way to calculate it .I want final result count.',
  'role': 'assistant',
  'name': 'code_executor_agent'},
 {'content': "Sure, we can use the permutations function from the itertools module of python to find the permutations of the word 'ALGEBRA'. \n\nHere is your python code:\n\n```python\nimport itertools\n\ndef count_permutations(word):\n    # Creating a set to store unique permutations\n    unique_permutations = set(itertools.permutations(word))\n    return len(unique_permutations)\n\nword = 'ALGEBRA'\nprint(count_permutations(word))\n```\nThis code will count the number of unique permutations of the word 'ALGEBRA'. It uses the itertools.permutations function to generate the permutations, and then a set is used to remove any duplicate permutations (since sets only contain unique elements). The len function is then used to count the number of elements in the set, giving the number of unique permutat

In [11]:
chat_result.chat_id

201114971642280354031000790733673483997

In [12]:
chat_result.cost

{'usage_including_cached_inference': {'total_cost': 0.10502999999999998,
  'gpt-4-0613': {'cost': 0.10502999999999998,
   'prompt_tokens': 2193,
   'completion_tokens': 654,
   'total_tokens': 2847}},
 'usage_excluding_cached_inference': {'total_cost': 0.10502999999999998,
  'gpt-4-0613': {'cost': 0.10502999999999998,
   'prompt_tokens': 2193,
   'completion_tokens': 654,
   'total_tokens': 2847}}}

In [13]:
chat_result.human_input

[]

In [14]:
chat_result.summary

''